In [ ]:
import sys
try:
    import kagglehub
    import warnings
    import os
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import torch
    import torch.nn as nn
    import torchvision.transforms.v2 as T
    from IPython.display import clear_output
    from PIL import Image
    from torch.utils.data import DataLoader
    from tqdm import tqdm
    from torch.optim import Optimizer
    from torchmetrics.classification import BinaryF1Score
    import os
    from sklearn.model_selection import train_test_split
    from torch.utils.data import Dataset, DataLoader
    import torch.nn.functional as F
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    from torch.nn.modules.linear import Linear
    from torch.nn.modules.flatten import Flatten
    from torch.nn.modules.activation import GELU
    from torch.nn.modules.batchnorm import BatchNorm2d
    from torch.optim.lr_scheduler import OneCycleLR
    from sklearn.metrics import f1_score
    import cv2
    import numpy as np
except ImportError as e:
    print(f"Error {e} ")
    !pip install kagglehub torch torchvision numpy pandas matplotlib scikit-learn tqdm pillow torchmetrics kagglehub

In [ ]:
@torch.inference_mode()
def predict_test(model, test_loader, device, threshold=0.5):
    model.eval()

    all_ids = []
    all_preds = []

    for x, ids in tqdm(test_loader):
        x = x.to(device)

        output = model(x)
        res = torch.sigmoid(output).squeeze(1)
        preds = (res > threshold).long()

        all_ids.extend(ids.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())

    submission = pd.DataFrame({
        "id": all_ids,
        "target_feature": all_preds
    })

    submission = submission.sort_values("id").reset_index(drop=True)# сортирует по возрастанию id удаляет старые индексы и созает новые
    submission.to_csv(f"{OUTPUTPATH}submission.csv", index=False)

    return submission